# 0. Setup

In [1]:
import ibis
from ibis import _
import pandas as pd
from utils.f_0_dirs import get_data_dirs
from linearmodels.panel.results import PanelEffectsResults
from f_7_run_panel import run_panel, ModelSpec, format_str, reindex_entity, add_fe

dirs = get_data_dirs(segment="model")
con = ibis.duckdb.connect(dirs.db_path, read_only=True)

# First difference the panel data

In [2]:
%%script fd sample
g_name = 'working_yearly_g'
n_name = 'working_yearly_n'
t_panel_g = con.table(g_name)
t_panel_n = con.table(n_name)

#--- #
t_panel = add_fe(t_panel_g, fe=['t', 'i', 'c'])
t_panel_sample = (
    t_panel
    .select([c for c in t_panel.columns if c.startswith('year_')])
    .order_by(ibis.random())
    .limit(5)
)
display(t_panel_sample.execute())

Couldn't find program: 'fd'


# 1a. LMM
- 26 seconds run nowadays estimating 2 models

In [ ]:
def_panel_name = "working_yearly_g"

# 3. Models: varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
models_1 = {
    'lmm_exog': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'fe': ['t', 'i', 'c'],
        'description': 'Strict exogeneity, structural form'
    },
    'lmm_instr': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'fe': ['t', 'i'],
        'Z': {
            'wg1_y': [
                'w2g1_k', 'w2g1_l',
                'w3g1_k', 'w3g1_l'
            ],
        },
        'description': 'Instrument endogenous effect, LMM'
    }
}
models_3 = {
    'lmm_3_rings': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'W': ['wg2_y', 'wg2_k', 'wg2_l', 'wg3_l', 'wg3_k', 'wg3_y'],
        'fe': ['t', 'i', 'c'],
        'description': 'Strict exogeneity, structural form',
        'include': True
    },
    'lmm_3_rings_instr': {
        'Y': 'y',
        'X': [
            'k', 'l',
            'wg1_y', 'wg1_k', 'wg1_l',
            'wg2_y', 'wg2_k', 'wg2_l',
            'wg3_y', 'wg3_k', 'wg3_l'
        ],
        'fe': ['i', 't'],
        'Z': {
            'wg1_y': ['w2g1_k', 'w2g1_l'],
            'wg2_y': ['w2g2_k', 'w2g2_l'],
            'wg3_y': ['w2g3_k', 'w2g3_l']
        },
        'description': '3-level regressors, squared instruments',
        'include': True
    },
    'lmm_3_rings_more_instr': {
        'Y': 'y',
        'X': [
            'k', 'l',
            'wg1_y', 'wg1_k', 'wg1_l',
            'wg2_y', 'wg2_k', 'wg2_l',
            'wg3_y', 'wg3_k', 'wg3_l'
        ],
        'fe': ['i', 'c', 't'],
        'Z': {
            'wg1_y': ['w2g1_k', 'w2g1_l', 'wg1wg2_k', 'wg1wg2_l'],#, 'wg1wg3_k', 'wg1wg3_l'],
            'wg2_y': ['w2g2_k', 'w2g2_l', 'wg2wg3_k', 'wg2wg3_l'],#, 'wg2wg1_k', 'wg2wg1_l'],
            'wg3_y': ['w2g3_k', 'w2g3_l', 'wg3wg1_k', 'wg3wg1_l']#, 'wg3wg2_k', 'wg3wg2_l']
        },
        'description': '3-level regressors, many instruments',
        'include': False
    }
}
models_fe = {
    'lmm_1_instr2_lnfe': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'fe': ['t', 'lnfe'],
        'Z': {
            'wg1_y': [
                'w2g1_k', 'w2g1_l'
            ]
        },
        'fe_type': 'g'
    },
    'lmm_1_instr4_lnfe': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'fe': ['t', 'lnfe'],
        'Z': {
            'wg1_y': [
                'w2g1_k', 'w2g1_l',
                'w3g1_k', 'w3g1_l',
                'w4g1_k', 'w4g1_l'
            ]
        },
        'fe_type': 'g'
    },
    'lmm_1_instr2_gnfe': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'fe': ['t', 'gnfe'],
        'Z': {
            'wg1_y': [
                'w2g1_k', 'w2g1_l'
            ]
        },
        'fe_type': 'g'
    },
    'lmm_1_instr4_gnfe': {
        'Y': 'y',
        'X': ['k', 'l', 'wg1_y', 'wg1_k', 'wg1_l'],
        'fe': ['t', 'gnfe'],
        'Z': {
            'wg1_y': [
                'w2g1_k', 'w2g1_l',
                'w3g1_k', 'w3g1_l',
                'w4g1_k', 'w4g1_l'
            ]
        },
        'fe_type': 'g'
    }
}

models = models_fe
out_name = "results_1c_lmm_nfe"
category = None

# 5m run for 3 complicated IV models
run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args: list[tuple[ModelSpec, str, str]] = []
for m_key, mod_obj in models.items():
    mod = ModelSpec(**mod_obj)
    if (not mod.include) or (category is not None and mod.category != category):
        print(f"Model '{m_key}' is excluded. Skipping.")
        continue
    worker_args.append((
        mod,
        mod.panel_name if mod.panel_name is not None else def_panel_name,
        m_key
    ))

# Refactor this so we open a blank new file,
# and then with each res, mod, model_name, we append to the file instead of writing all at once at the end.
# This way, if a model fails or I interrupt, we still have the results of the previous models saved.
with open(dirs.output_dir / f"{out_name}.txt", "w") as f:
    for args in worker_args:
        mod, _, model_name = args
        res, beta, effects_dict = run_panel(args)
        if res is None:
            print(f"Model '{model_name}' failed. Skipping.")
            continue
        output_str = format_str((res, mod, model_name))     # type: ignore
        f.write(output_str + "\n" + "=" * 80 + "\n" + "=" * 80 + "\n")
        run_res_series.append((res, mod, model_name))       # type: ignore

model_count = len(run_res_series)
if model_count:
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}. Saved to {out_name}.txt")

Running model 'lmm_1_instr2_lnfe' as linearmodels panel IV, formula: `(i-wg1)_y` ~ `(i-wg1)_k` + `(i-wg1)_l` + `(i-wg1)wg1_k` + `(i-wg1)wg1_l` + year_2019 + year_2018 + year_2013 + year_2010 + year_2009 + year_2012 + year_2007 + year_2022 + year_2006 + year_2017 + year_2016 + year_2020 + year_2008 + year_2011 + year_2023 + year_2024 + year_2021 + year_2014 + year_2015 + [`(i-wg1)wg1_y` ~ `(i-wg1)w2g1_k` + `(i-wg1)w2g1_l`]
J-statistic (rej if overidentified): 52.52, p-value: 0.000
✅ Model 'lmm_1_instr2_lnfe' estimated: (i-wg1)_k=0.328, (i-wg1)_l=0.622, (i-wg1)wg1_y=-20.977, (i-wg1)wg1_k=7.106, (i-wg1)wg1_l=12.888
Running model 'lmm_1_instr4_lnfe' as linearmodels panel IV, formula: `(i-wg1)_y` ~ `(i-wg1)_k` + `(i-wg1)_l` + `(i-wg1)wg1_k` + `(i-wg1)wg1_l` + year_2019 + year_2018 + year_2016 + year_2008 + year_2011 + year_2013 + year_2017 + year_2006 + year_2009 + year_2010 + year_2022 + year_2007 + year_2023 + year_2021 + year_2024 + year_2012 + year_2015 + year_2014 + year_2020 + [`(i-wg

# 2. Distance decay model

$$
\begin{align*}
z_{it} &= \alpha_i + \gamma_t + \rho \sum_{j \neq i} f(d_{ij}) \cdot z_{jt} + \epsilon_{it}   \\
y_{it} &= \alpha_i + \gamma_t + \beta_1 k_{it} + \beta_2 l_{it} + \rho \sum_{j \neq i} w_{ij} z_{jt} + \epsilon_{it}
\end{align*}
$$

In [ ]:
from linearmodels.panel.results import PanelEffectsResults
from f_7_run_panel import run_panel, ModelSpec, format_str

def_panel_name = "working_yearly_n"

# 3. Models: varying Y (gva1 vs gva2) and K (fixed_total vs tangibles)
m_models = {
    # Basic models
    'dd1': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l']
        },
        'fe': ['i', 't'],
        'description': 'Distance 1 model',
        'panel_name': 'working_yearly_n',
        'include': False
    },
    'dd2': {
        'Y': 'y',
        'X': ['k', 'l', 'wd2_y', 'wd2_k', 'wd2_l'],
        'Z': {
            'wd2_y': ['w2d2_k', 'w2d2_l']
        },
        'fe': ['i', 't'],
        'description': 'Distance 2 model',
        'panel_name': 'working_yearly_n',
        'include': False
    },
    'dd3': {
        'Y': 'y',
        'X': ['k', 'l', 'wd3_y', 'wd3_k', 'wd3_l'],
        'Z': {
            'wd3_y': ['w2d3_k', 'w2d3_l']
        },
        'fe': ['i', 't'],
        'description': 'Distance 3 model',
        'panel_name': 'working_yearly_n',
        'include': False
    },

    # Models excluding firms which share a location
    'dd1-no-zeroes': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l']
        },
        'fe': ['i', 't'],
        'description': 'Distance 1 model, firms which share a location excluded',
        'panel_name': 'working_yearly_no',
        'include': False
    },
    'dd2-no-zeroes': {
        'Y': 'y',
        'X': ['k', 'l', 'wd2_y', 'wd2_k', 'wd2_l'],
        'Z': {
            'wd2_y': ['w2d2_k', 'w2d2_l']
        },
        'fe': ['i', 't'],
        'description': 'Distance 2 model, firms which share a location excluded',
        'panel_name': 'working_yearly_no',
        'include': False
    },
    'dd3-no-zeroes': {
        'Y': 'y',
        'X': ['k', 'l', 'wd3_y', 'wd3_k', 'wd3_l'],
        'Z': {
            'wd3_y': ['w2d3_k', 'w2d3_l']
        },
        'fe': ['i', 't'],
        'description': 'Distance 3 model, firms which share a location excluded',
        'panel_name': 'working_yearly_no',
        'include': False
    },

    # Models with more instruments (high-order lags)
    'dd1-3lag': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l', 'w3d1_k', 'w3d1_l']
        },
        'fe': ['i', 't'],
        'description': '3rd-order spatial lag model',
        'panel_name': 'working_yearly_n',
        'include': False
    },
    'dd1-3lag-no-zeroes': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l', 'w3d1_k', 'w3d1_l']
        },
        'fe': ['i', 't'],
        'description': '3rd-order spatial lag model',
        'panel_name': 'working_yearly_no',
        'include': False
    },

    # Network fixed effects
    'dd1-lnfe': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l']
        },
        'fe': ['t', 'lnfe'],
        'description': 'Distance 1 model',
        'panel_name': 'working_yearly_n',
        'category': 'nfe'
    },
    'dd1-gnfe': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l']
        },
        'fe': ['t', 'gnfe'],
        'panel_name': 'working_yearly_n',
        'category': 'nfe'
    },
    'dd1-noz-lnfe': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l']
        },
        'fe': ['t', 'lnfe'],
        'panel_name': 'working_yearly_no',
        'category': 'nfe'
    },
    'dd1-noz-gnfe': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l']
        },
        'fe': ['t', 'gnfe'],
        'panel_name': 'working_yearly_no',
        'category': 'nfe'
    },
    'dd1-3lag-noz-lnfe': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l', 'w3d1_k', 'w3d1_l']
        },
        'fe': ['t', 'lnfe'],
        'panel_name': 'working_yearly_no',
        'category': 'nfe'
    },
    'dd1-4lag-noz-gnfe': {
        'Y': 'y',
        'X': ['k', 'l', 'wd1_y', 'wd1_k', 'wd1_l'],
        'Z': {
            'wd1_y': ['w2d1_k', 'w2d1_l', 'w3d1_k', 'w3d1_l', 'w4d1_k', 'w4d1_l']
        },
        'fe': ['t', 'gnfe'],
        'panel_name': 'working_yearly_no',
        'category': 'nfe'
    }
}

models = m_models
out_name = "results_2c_combined_nfes"
category = 'nfe'

run_res_series: list[tuple[PanelEffectsResults, ModelSpec, str]] = []
worker_args: list[tuple[ModelSpec, str, str]] = []
for m_key, mod_obj in models.items():
    mod = ModelSpec(**mod_obj)
    if (not mod.include) or (category is not None and mod.category != category):
        print(f"Model '{m_key}' is excluded. Skipping.")
        continue
    worker_args.append((
        mod,
        mod.panel_name if mod.panel_name is not None else def_panel_name,
        m_key
    ))

# Refactor this so we open a blank new file,
# and then with each res, mod, model_name, we append to the file instead of writing all at once at the end.
# This way, if a model fails or I interrupt, we still have the results of the previous models saved.
with open(dirs.output_dir / f"{out_name}.txt", "w") as f:
    for args in worker_args:
        mod, _, model_name = args
        res, beta, effects_dict = run_panel(args)
        if res is None:
            print(f"Model '{model_name}' failed. Skipping.")
            continue
        output_str = format_str((res, mod, model_name))     # type: ignore
        f.write(output_str + "\n" + "=" * 80 + "\n" + "=" * 80 + "\n")
        run_res_series.append((res, mod, model_name))       # type: ignore

model_count = len(run_res_series)
if model_count:
    print(f"Panel regressions complete. {model_count} model{'' if model_count == 1 else 's'}. Saved to {out_name}.txt")

Model 'dd1' is excluded. Skipping.
Model 'dd2' is excluded. Skipping.
Model 'dd3' is excluded. Skipping.
Model 'dd1-no-zeroes' is excluded. Skipping.
Model 'dd2-no-zeroes' is excluded. Skipping.
Model 'dd3-no-zeroes' is excluded. Skipping.
Model 'dd1-3lag' is excluded. Skipping.
Model 'dd1-3lag-no-zeroes' is excluded. Skipping.
Running model 'dd1-lnfe' as linearmodels panel IV, formula: `(i-wd1)_y` ~ `(i-wd1)_k` + `(i-wd1)_l` + `(i-wd1)wd1_k` + `(i-wd1)wd1_l` + year_2019 + year_2018 + year_2016 + year_2022 + year_2007 + year_2014 + year_2015 + year_2013 + year_2023 + year_2021 + year_2024 + year_2008 + year_2017 + year_2006 + year_2009 + year_2010 + year_2012 + year_2011 + year_2020 + [`(i-wd1)wd1_y` ~ `(i-wd1)w2d1_k` + `(i-wd1)w2d1_l`]
J-statistic (rej if overidentified): 13.44, p-value: 0.000
✅ Model 'dd1-lnfe' estimated: (i-wd1)_k=0.388, (i-wd1)_l=0.579, (i-wd1)wd1_y=1.575, (i-wd1)wd1_k=-0.510, (i-wd1)wd1_l=-1.035
Running model 'dd1-gnfe' as linearmodels panel IV, formula: `(i-vd1)_